## Milestone 4: A/B Test Evaluation & Experiment Diagnostics

### 1. Experiment Overview

This notebook evaluates the homepage A/B test to determine whether
the new homepage variant delivers a statistically and practically
meaningful improvement in signup conversion.

The analysis covers:

- Dataset validation
- Sample Ratio Mismatch (SRM)
- Conversion rate and statistical significance
- 95% confidence intervals
- Minimum Detectable Effect (MDE) evaluation
- 30-day revenue guardrail
- Novelty effect and temporal stability
- Device-level performance analysis
- Final experiment recommendation

In [1]:
import pandas as pd


df = pd.read_csv("../data/homepage_ab_test.csv")

# Dataset dimensions
print("Dataset Shape:", df.shape)

# Column names and data types
print("\nColumn Information:")
print(df.info())

# First five rows
print("\nFirst Five Rows:")
display(df.head())

# Missing values
print("\nMissing Values:")
display(df.isnull().sum())

# Experiment group distribution
print("\nExperiment Group Distribution:")
display(df["arm"].value_counts())

# Unique values in categorical columns
for col in df.select_dtypes(include="object").columns:
    print(f"\n{col}:")
    print(df[col].value_counts().head(10))

Dataset Shape: (50000, 7)

Column Information:
<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   user_id             50000 non-null  str    
 1   arm                 50000 non-null  str    
 2   signup_date         50000 non-null  str    
 3   signup_completed    50000 non-null  int64  
 4   revenue_30d         50000 non-null  float64
 5   device              50000 non-null  str    
 6   acquisition_source  50000 non-null  str    
dtypes: float64(1), int64(1), str(5)
memory usage: 2.7 MB
None

First Five Rows:


,user_id,arm,signup_date,signup_completed,revenue_30d,device,acquisition_source
0,U00001,control,2024-05-09,1,98.48,mobile,organic
1,U00002,variant,2024-05-04,0,0.00,mobile,organic
2,U00003,variant,2024-05-06,0,0.00,desktop,organic
3,U00004,variant,2024-05-08,0,0.00,mobile,organic
4,U00005,control,2024-05-10,0,0.00,mobile,social



Missing Values:


user_id               0
arm                   0
signup_date           0
signup_completed      0
revenue_30d           0
device                0
acquisition_source    0
dtype: int64


Experiment Group Distribution:


arm
control    25053
variant    24947
Name: count, dtype: int64


user_id:
user_id
U00001    1
U00002    1
U00003    1
U00004    1
U00005    1
U00006    1
U00007    1
U00008    1
U00009    1
U00010    1
Name: count, dtype: int64

arm:
arm
control    25053
variant    24947
Name: count, dtype: int64

signup_date:
signup_date
2024-05-13    3650
2024-05-14    3636
2024-05-09    3629
2024-05-10    3624
2024-05-01    3616
2024-05-11    3564
2024-05-02    3557
2024-05-04    3552
2024-05-07    3548
2024-05-05    3544
Name: count, dtype: int64

device:
device
mobile     25029
desktop    24971
Name: count, dtype: int64

acquisition_source:
acquisition_source
organic        17361
paid_search    14983
social         10080
referral        7576
Name: count, dtype: int64


/var/folders/b8/mkpb1vhs3w9gtk0mssh38jgr0000gn/T/ipykernel_60702/3139241152.py:26: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


### 2. Sample Ratio Mismatch (SRM) Test

The experiment was designed with a 50/50 allocation
between the control and variant groups.

A Chi-square goodness-of-fit test is used to verify
whether the observed sample distribution differs
significantly from the expected allocation.

Significance level: 0.05

In [2]:
from scipy.stats import chisquare

# Observed sample counts
observed = df["arm"].value_counts().reindex(
    ["control", "variant"]
).values

# Expected sample counts (50/50 allocation)
expected = [len(df) / 2, len(df) / 2]

# Chi-square goodness-of-fit test
chi_stat, p_value = chisquare(
    f_obs=observed,
    f_exp=expected
)

# Display results
print("SRM Test Results")
print("----------------")
print("Observed Counts:", observed)
print("Expected Counts:", expected)
print(f"Chi-square Statistic: {chi_stat:.4f}")
print(f"P-value: {p_value:.4f}")

alpha = 0.05

if p_value > alpha:
    print("\nConclusion: No statistically significant SRM detected.")
else:
    print("\nConclusion: Sample Ratio Mismatch detected.")

SRM Test Results
----------------
Observed Counts: [25053 24947]
Expected Counts: [25000.0, 25000.0]
Chi-square Statistic: 0.2247
P-value: 0.6355

Conclusion: No statistically significant SRM detected.


### 3. Primary Metric – Signup Conversion Analysis

The primary experiment metric is signup conversion rate.

A two-sample proportion Z-test is used to evaluate
whether the conversion rates differ significantly
between the control and variant groups.

The analysis also calculates the absolute conversion
lift and its 95% Wald confidence interval.

The business requires a minimum conversion lift
of +0.50 percentage points.

In [3]:
from statsmodels.stats.proportion import (
    proportions_ztest,
    confint_proportions_2indep
)

# Separate experiment groups
control = df[df["arm"] == "control"]
variant = df[df["arm"] == "variant"]

# Sample sizes
n_control = len(control)
n_variant = len(variant)

# Conversion counts
conv_control = control["signup_completed"].sum()
conv_variant = variant["signup_completed"].sum()

# Conversion rates
rate_control = conv_control / n_control
rate_variant = conv_variant / n_variant

# Absolute conversion lift (Variant - Control)
lift = rate_variant - rate_control

# Two-sample proportion Z-test
z_stat, p_value = proportions_ztest(
    count=[conv_variant, conv_control],
    nobs=[n_variant, n_control],
    alternative="two-sided"
)

# 95% Wald confidence interval for the difference
ci_low, ci_high = confint_proportions_2indep(
    count1=conv_variant,
    nobs1=n_variant,
    count2=conv_control,
    nobs2=n_control,
    method="wald",
    compare="diff",
    alpha=0.05
)

# Display results
print("SIGNUP CONVERSION ANALYSIS")
print("-" * 40)

print(f"Control users: {n_control:,}")
print(f"Control conversions: {conv_control:,}")
print(f"Control conversion rate: {rate_control:.4%}")

print(f"\nVariant users: {n_variant:,}")
print(f"Variant conversions: {conv_variant:,}")
print(f"Variant conversion rate: {rate_variant:.4%}")

print(f"\nAbsolute lift: {lift:.4%}")
print(f"Absolute lift (pp): {lift * 100:.4f} pp")

print(f"\nZ-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.6f}")

print("\n95% Wald Confidence Interval:")
print(f"Lower bound: {ci_low * 100:.4f} pp")
print(f"Upper bound: {ci_high * 100:.4f} pp")

# Statistical significance
if p_value < 0.05:
    print("\nStatistical conclusion: Significant difference detected.")
else:
    print("\nStatistical conclusion: No significant difference detected.")

# Business MDE evaluation
mde = 0.005

print("\nMDE Evaluation:")
print(f"Required MDE: {mde * 100:.2f} pp")

if lift >= mde:
    print("Observed lift meets or exceeds the MDE.")
else:
    print("Observed lift does NOT meet the MDE.")

SIGNUP CONVERSION ANALYSIS
----------------------------------------
Control users: 25,053
Control conversions: 712
Control conversion rate: 2.8420%

Variant users: 24,947
Variant conversions: 802
Variant conversion rate: 3.2148%

Absolute lift: 0.3728%
Absolute lift (pp): 0.3728 pp

Z-statistic: 2.4326
P-value: 0.014990

95% Wald Confidence Interval:
Lower bound: 0.0724 pp
Upper bound: 0.6733 pp

Statistical conclusion: Significant difference detected.

MDE Evaluation:
Required MDE: 0.50 pp
Observed lift does NOT meet the MDE.


### 4. Revenue Guardrail – 30-Day ARPU

The experiment's secondary metric is 30-day revenue
per randomized user.

Revenue is compared between the control and variant
groups using Welch's independent two-sample t-test.

The test uses equal_var=False because equal population
variances are not assumed.

The analysis evaluates whether the conversion improvement
is accompanied by a deterioration in revenue per user.

In [4]:
from scipy.stats import ttest_ind

# Extract 30-day revenue for each experiment group
ctrl_rev = control["revenue_30d"]
trt_rev = variant["revenue_30d"]

# Calculate ARPU
arpu_control = ctrl_rev.mean()
arpu_variant = trt_rev.mean()

# Absolute ARPU difference
arpu_diff = arpu_variant - arpu_control

# Welch's independent two-sample t-test
t_stat, revenue_p_value = ttest_ind(
    ctrl_rev,
    trt_rev,
    equal_var=False,
    alternative="two-sided"
)

# Display results
print("30-DAY REVENUE GUARDRAIL")
print("-" * 40)

print(f"Control total revenue: ${ctrl_rev.sum():,.2f}")
print(f"Variant total revenue: ${trt_rev.sum():,.2f}")

print(f"\nControl ARPU: ${arpu_control:.4f}")
print(f"Variant ARPU: ${arpu_variant:.4f}")

print(f"\nAbsolute ARPU difference: ${arpu_diff:.4f}")

print(f"\nWelch's t-statistic: {t_stat:.4f}")
print(f"P-value: {revenue_p_value:.6f}")

alpha = 0.05

if revenue_p_value < alpha:
    print("\nConclusion: Statistically significant revenue difference.")
else:
    print("\nConclusion: No statistically significant revenue difference.")

if arpu_diff < 0:
    print("Revenue direction: Variant has lower ARPU.")
else:
    print("Revenue direction: Variant has higher ARPU.")

30-DAY REVENUE GUARDRAIL
----------------------------------------
Control total revenue: $83,011.10
Variant total revenue: $72,753.34

Control ARPU: $3.3134
Variant ARPU: $2.9163

Absolute ARPU difference: $-0.3971

Welch's t-statistic: 2.1411
P-value: 0.032274

Conclusion: Statistically significant revenue difference.
Revenue direction: Variant has lower ARPU.


### 5. Novelty Effect – Temporal Stability Analysis

This analysis evaluates whether the homepage variant's
conversion improvement remains stable over time.

The experiment is divided into Week 1 and Week 2
using the signup_date column.

Conversion rates and absolute lift are compared
between the two periods to investigate potential
novelty effects.

In [5]:
# Convert signup_date to datetime
df["signup_date"] = pd.to_datetime(df["signup_date"])

# Inspect experiment date range
print("Experiment Start Date:", df["signup_date"].min())
print("Experiment End Date:", df["signup_date"].max())

# Inspect number of unique dates
print("\nNumber of Unique Dates:", df["signup_date"].nunique())

# Daily user distribution
daily_users = df.groupby(
    df["signup_date"].dt.date
).size()

print("\nDaily User Distribution:")
display(daily_users)

Experiment Start Date: 2024-05-01 00:00:00
Experiment End Date: 2024-05-14 00:00:00

Number of Unique Dates: 14

Daily User Distribution:


signup_date
2024-05-01    3616
2024-05-02    3557
2024-05-03    3506
2024-05-04    3552
2024-05-05    3544
2024-05-06    3504
2024-05-07    3548
2024-05-08    3540
2024-05-09    3629
2024-05-10    3624
2024-05-11    3564
2024-05-12    3530
2024-05-13    3650
2024-05-14    3636
dtype: int64

In [6]:
# Define Week 1 and Week 2
week1 = df[df["signup_date"] < "2024-05-08"]

week2 = df[df["signup_date"] >= "2024-05-08"]

# Function to calculate weekly conversion metrics
def calculate_weekly_metrics(data, week_name):

    control_week = data[data["arm"] == "control"]
    variant_week = data[data["arm"] == "variant"]

    n_control = len(control_week)
    n_variant = len(variant_week)

    conv_control = control_week["signup_completed"].sum()
    conv_variant = variant_week["signup_completed"].sum()

    rate_control = conv_control / n_control
    rate_variant = conv_variant / n_variant

    lift = rate_variant - rate_control

    return {
        "Week": week_name,
        "Control Users": n_control,
        "Variant Users": n_variant,
        "Control Conversions": conv_control,
        "Variant Conversions": conv_variant,
        "Control Conversion Rate": rate_control,
        "Variant Conversion Rate": rate_variant,
        "Absolute Lift (pp)": lift * 100
    }


# Calculate weekly results
week1_results = calculate_weekly_metrics(week1, "Week 1")
week2_results = calculate_weekly_metrics(week2, "Week 2")

# Combine results
weekly_results = pd.DataFrame([
    week1_results,
    week2_results
])

display(weekly_results)

,Week,Control Users,Variant Users,Control Conversions,Variant Conversions,Control Conversion Rate,Variant Conversion Rate,Absolute Lift (pp)
0,Week 1,12428,12399,362,420,0.029128,0.033874,0.474592
1,Week 2,12625,12548,350,382,0.027723,0.030443,0.272033


### 6. Device-Level Performance Analysis

This section evaluates whether the homepage variant
performs differently across Desktop and Mobile users.

Conversion rates and absolute conversion lift are
calculated separately for each device segment.

The analysis aims to identify segment-level performance
divergence that may be hidden by the overall experiment results.

In [7]:
# Function to calculate device-level conversion metrics

def calculate_device_metrics(data, device_name):

    control_device = data[
        (data["arm"] == "control") &
        (data["device"] == device_name)
    ]

    variant_device = data[
        (data["arm"] == "variant") &
        (data["device"] == device_name)
    ]

    n_control = len(control_device)
    n_variant = len(variant_device)

    conv_control = control_device["signup_completed"].sum()
    conv_variant = variant_device["signup_completed"].sum()

    rate_control = conv_control / n_control
    rate_variant = conv_variant / n_variant

    lift = rate_variant - rate_control

    return {
        "Device": device_name,
        "Control Users": n_control,
        "Variant Users": n_variant,
        "Control Conversions": conv_control,
        "Variant Conversions": conv_variant,
        "Control Conversion Rate": rate_control,
        "Variant Conversion Rate": rate_variant,
        "Absolute Lift (pp)": lift * 100
    }


# Calculate metrics for each device
desktop_results = calculate_device_metrics(df, "desktop")
mobile_results = calculate_device_metrics(df, "mobile")

# Combine results
device_results = pd.DataFrame([
    desktop_results,
    mobile_results
])

display(device_results)

,Device,Control Users,Variant Users,Control Conversions,Variant Conversions,Control Conversion Rate,Variant Conversion Rate,Absolute Lift (pp)
0,desktop,12553,12418,426,535,0.033936,0.043083,0.914651
1,mobile,12500,12529,286,267,0.022880,0.021311,-0.156944


### Device-Level Interpretation

The experiment showed divergent conversion outcomes across device segments.

- **Desktop:** Conversion increased from 3.39% in Control to 4.31% in Variant, representing a positive lift of approximately 0.91 percentage points.
- **Mobile:** Conversion declined from 2.29% in Control to 2.13% in Variant, representing a negative lift of approximately 0.16 percentage points.

The overall positive conversion result appears to be driven primarily by Desktop users. The negative Mobile result raises a potential usability or experience concern that should be investigated before a full rollout.

These segment-level differences are descriptive; further statistical testing would be needed to establish whether the treatment effect differs significantly by device.

### Statistical Test: Device-Level Treatment Interaction

A logistic regression model with an interaction between experiment arm and device is used to assess whether the treatment effect differs significantly between Desktop and Mobile users.

In [8]:
import statsmodels.formula.api as smf

device_model = smf.logit(
    "signup_completed ~ C(arm, Treatment(reference='control')) * C(device, Treatment(reference='desktop'))",
    data=df
).fit()

print(device_model.summary())

Optimization terminated successfully.
         Current function value: 0.134408
         Iterations 8
                           Logit Regression Results                           
Dep. Variable:       signup_completed   No. Observations:                50000
Model:                          Logit   Df Residuals:                    49996
Method:                           MLE   Df Model:                            3
Date:                Thu, 26 Nov 2026   Pseudo R-squ.:                0.009622
Time:                        03:49:42   Log-Likelihood:                -6720.4
converged:                       True   LL-Null:                       -6785.7
Covariance Type:            nonrobust   LLR p-value:                 4.045e-28
                                                                                                            coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------

### Device Interaction Test: Interpretation

A logistic regression model was fitted with experiment arm, device, and their interaction.

The interaction between Variant and Mobile was negative and statistically significant (coefficient = -0.3208, p = 0.003, 95% CI: [-0.534, -0.108]).

This indicates that the treatment effect differs significantly between Desktop and Mobile users.

The positive Desktop conversion lift (+0.91 percentage points) and negative Mobile conversion lift (-0.16 percentage points) are therefore not merely descriptive differences. The interaction test provides statistical evidence of device-level treatment heterogeneity.

A universal rollout is not recommended without further investigation of the Mobile experience.